# Hybrid DeBERTa + Qwen 14B Inference
- Uses DeBERTa ensemble for all predictions
- Identifies hard examples (top 10% by std across seeds)
- Runs Qwen 14B inference only on hard examples
- Blends: 0.5 * deberta_prob + 0.5 * qwen_prob for hard examples

## 1. Load DeBERTa Predictions

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.special import softmax

SEEDS = [42, 123]
HARD_EXAMPLE_RATIO = 0.10

In [ ]:
seed_preds = []
for seed in SEEDS:
    df_seed = pd.read_csv(f'test_preds_seed_{seed}.csv')
    seed_preds.append(df_seed[f'pred_seed_{seed}'].values)
    print(f"Loaded seed {seed}: {len(df_seed)} predictions")

seed_preds = np.array(seed_preds)
print(f"\nShape: {seed_preds.shape}")

## 2. Calculate Mean and Std, Identify Hard Examples

In [ ]:
deberta_mean = seed_preds.mean(axis=0)
deberta_std = seed_preds.std(axis=0)

print(f"Mean predictions - min: {deberta_mean.min():.4f}, max: {deberta_mean.max():.4f}")
print(f"Std - min: {deberta_std.min():.4f}, max: {deberta_std.max():.4f}, mean: {deberta_std.mean():.4f}")

In [ ]:
n_hard = int(len(deberta_mean) * HARD_EXAMPLE_RATIO)
hard_indices = np.argsort(deberta_std)[-n_hard:]

print(f"Total examples: {len(deberta_mean)}")
print(f"Hard examples (top {HARD_EXAMPLE_RATIO*100}% by std): {n_hard}")
print(f"Std threshold: {deberta_std[hard_indices].min():.4f}")
print(f"\nHard example indices: {sorted(hard_indices[:10])}...")

## 3. Prepare Test Data and Hard Examples for 14B

In [ ]:
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
df_test = pd.read_csv(test_path)
print(f"Test data shape: {df_test.shape}")
df_test.head()

In [ ]:
df_test['deberta_mean'] = deberta_mean
df_test['deberta_std'] = deberta_std
df_test['is_hard'] = False
df_test.loc[hard_indices, 'is_hard'] = True

print(f"\nHard examples: {df_test['is_hard'].sum()}")
df_test[df_test['is_hard']].head()

In [ ]:
df_hard = df_test[df_test['is_hard']].copy()
print(f"Hard examples for 14B inference: {len(df_hard)}")
print(f"\nStd stats for hard examples:")
print(df_hard['deberta_std'].describe())

## 4. Setup Qwen 14B Model

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

In [ ]:
import torch
import vllm
from vllm.lora.request import LoRARequest
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor

MODEL_NAME = "/kaggle/input/qwen2.5/transformers/14b-instruct-gptq-int4/1"
LORA_PATH = "/kaggle/input/lora_14b_gptq_1epoch_r32/keras/default/1"

os.environ["VLLM_USE_V1"] = "0"

In [ ]:
llm = vllm.LLM(
    MODEL_NAME,
    quantization='gptq',
    tensor_parallel_size=torch.cuda.device_count(),
    gpu_memory_utilization=0.98,
    trust_remote_code=True,
    dtype="half",
    enforce_eager=True,
    max_model_len=4096,
    disable_log_stats=True,
    enable_prefix_caching=True,
    enable_lora=True,
    max_lora_rank=32
)
llm

In [ ]:
tokenizer = llm.get_tokenizer()
mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])

## 5. Create Prompts for Hard Examples

In [ ]:
SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

prompts = []
for i, row in df_hard.iterrows():
    text = f"""
r/{row['subreddit']}
Rule: {row['rule']}

1) {row['positive_example_1']}
Violation: Yes

2) {row['positive_example_2']}
Violation: Yes

3) {row['negative_example_1']}
Violation: No

4) {row['negative_example_2']}
Violation: No

5) {row['body']}
"""
    
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    ) + "Answer:"
    prompts.append(prompt)

print(f"Created {len(prompts)} prompts for hard examples")
print(f"\nExample prompt:\n{prompts[0][:500]}...")

## 6. Run 14B Inference on Hard Examples Only

In [ ]:
outputs = llm.generate(
    prompts,
    vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    ),
    use_tqdm=True,
    lora_request=LoRARequest("default", 1, LORA_PATH)
)
print(f"Generated {len(outputs)} outputs")

In [ ]:
logprobs = [
    {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
    for out in outputs
]

print(f"Extracted logprobs for {len(logprobs)} examples")
print(f"\nExample logprobs: {logprobs[0]}")

## 7. Convert Logprobs to Probabilities

In [ ]:
logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
print(f"Logit matrix shape: {logit_matrix.shape}")
logit_matrix.head()

In [ ]:
qwen_probs = logit_matrix.apply(lambda x: softmax(x.values), axis=1, result_type="expand")
qwen_probs.columns = ['Yes', 'No']
qwen_probs['qwen_prob'] = qwen_probs['Yes']

print(f"Qwen probabilities shape: {qwen_probs.shape}")
print(f"\nStats:")
print(qwen_probs['qwen_prob'].describe())
qwen_probs.head()

## 8. Blend Predictions for Hard Examples

In [ ]:
df_hard['qwen_prob'] = qwen_probs['qwen_prob'].values
df_hard['blended_prob'] = 0.5 * df_hard['deberta_mean'] + 0.5 * df_hard['qwen_prob']

print(f"Blended {len(df_hard)} hard examples")
print(f"\nComparison:")
print(df_hard[['row_id', 'deberta_mean', 'deberta_std', 'qwen_prob', 'blended_prob']].head(10))

## 9. Generate Final Submission

In [ ]:
df_test['final_pred'] = df_test['deberta_mean']

for idx, row in df_hard.iterrows():
    df_test.loc[idx, 'final_pred'] = row['blended_prob']

print(f"Final predictions created")
print(f"Using blended predictions for {df_test['is_hard'].sum()} hard examples")
print(f"Using DeBERTa predictions for {(~df_test['is_hard']).sum()} easy examples")

In [ ]:
submission = df_test[['row_id', 'final_pred']].copy()
submission.columns = ['row_id', 'rule_violation']
submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"\nSubmission stats:")
print(submission['rule_violation'].describe())
submission.head(10)

In [ ]:
!head submission.csv

## 10. Analysis

In [ ]:
print("Hard Examples - Prediction Changes:")
df_hard['diff'] = df_hard['blended_prob'] - df_hard['deberta_mean']
print(f"\nMean change: {df_hard['diff'].mean():.4f}")
print(f"Std of change: {df_hard['diff'].std():.4f}")
print(f"Max increase: {df_hard['diff'].max():.4f}")
print(f"Max decrease: {df_hard['diff'].min():.4f}")

print("\nTop 5 examples where Qwen disagreed most with DeBERTa:")
print(df_hard.nlargest(5, 'diff')[['row_id', 'deberta_mean', 'qwen_prob', 'blended_prob', 'diff']])